In [0]:
CREATE OR REPLACE TABLE gold_lemonade_pl AS
WITH waterfall AS (
    SELECT
        1 AS sort_order,
        '1. Net Revenue' AS line_item,
        COALESCE((SELECT SUM(total_amount_usd) FROM default.silver_lemonade_spend), 0) AS amount_usd
    UNION ALL
    SELECT
        2 AS sort_order,
        '2. COGS (Raw Materials)' AS line_item,
        -COALESCE((SELECT SUM(clean_amount_usd) FROM silver_lemonade_spend_v2 WHERE category = 'Raw_Materials'), 0) AS amount_usd
    UNION ALL
    SELECT
        3 AS sort_order,
        '3. OPEX (Marketing & Misc)' AS line_item,
        -COALESCE((SELECT SUM(clean_amount_usd) FROM silver_lemonade_spend_v2 WHERE category != 'Raw_Materials'), 0) AS amount_usd
)
SELECT
    sort_order,
    line_item,
    amount_usd
FROM waterfall

UNION ALL

SELECT
    4 AS sort_order,
    '4. Operating Profit (EBIT)' AS line_item,
    SUM(amount_usd) AS amount_usd
FROM waterfall;

In [0]:
CREATE OR REPLACE TABLE gold_lemonade_working_capital AS
SELECT
    payment_terms,
    COUNT(*) AS total_transactions,
    COUNT(clean_amount_usd) AS priced_transactions,
    COUNT(*) - COUNT(clean_amount_usd) AS missing_amount_count,
    COALESCE(SUM(clean_amount_usd), CAST(0.00 AS DECIMAL(10, 2))) AS total_spend_usd
FROM default.silver_lemonade_spend_v2
GROUP BY payment_terms
ORDER BY total_spend_usd DESC;

SELECT * FROM gold_lemonade_working_capital;